# Tokenization

In [1]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')

# Load model and tokenizer
model = AutoModelForCausalLM.from_pretrained(
    "microsoft/Phi-3-mini-4k-instruct", 
    device_map = device, 
    torch_dtype='auto', 
    trust_remote_code=True, 
)
tokenizer = AutoTokenizer.from_pretrained('microsoft/Phi-3-mini-4k-instruct')

W0729 17:35:08.536000 3862 site-packages/torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.
`flash-attention` package not found, consider installing for better performance: No module named 'flash_attn'.
Current `flash-attention` does not support `window_size`. Either upgrade or use `attn_implementation='eager'`.


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


In [98]:
prompt = "Write an email apologizing for Sarah for the tragic car accident."
# Tokenize input prompt
inputs = tokenizer(prompt, return_tensors='pt').to('mps')

# Generate the text
generation_output = model.generate(
    input_ids = inputs['input_ids'], 
    attention_mask=inputs['attention_mask'],
    max_new_tokens=20
)

# Print the output
result = tokenizer.decode(generation_output[0])
print(result)

Write an email apologizing for Sarah for the tragic car accident.

Subject: My Deepest Apologies for the Tragic Accident

Dear


In [23]:
print(inputs.input_ids)
for id in input_ids.input_ids[0]:
    print(tokenizer.decode(id))

tensor([[14350,   385,  4876, 27746,  5281,   363, 19235,   363,   278, 25305,
           293,  1559, 11423, 29889]], device='mps:0')
Write
an
email
apolog
izing
for
Sarah
for
the
trag
ic
car
accident
.


In [34]:
generation_output

tensor([[14350,   385,  4876, 27746,  5281,   363, 19235,   363,   278, 25305,
           293,  1559, 11423, 29889,    13,    13, 20622, 29901,  1619, 21784,
           342,  6225, 11763,   363,   278,   323,  1431,   293,  4831,  1693,
            13,    13, 29928,   799]], device='mps:0')

### Four Notable Tokenizers:
1. Word Tokens
2. Subword Tokens
3. Character Tokens
4. Byte Tokens(tokenization-free)

## Comparing Trained LLM Tokenizers

In [87]:
text = """
English and CAPITILIZATION

🎵 鸟
show_tokens False None elif == >= else: two tabs:"  " Three tabs: "   "

12.0*50=600. 
"""
code_text = """
def add_numbers(a, b):

....# Add the two numbers `a` and `b`.

....return a + b
"""

In [66]:
colors_list = [
    '102;194;165', '252;141;98', '141;160;203', 
    '231;138;195', '166;216;84', '255;217;47'
]

def show_tokens(sentence, tokenizer_name):
    tokenizer = AutoTokenizer.from_pretrained(tokenizer_name)
    token_ids = tokenizer(sentence).input_ids
    for idx, t in enumerate(token_ids):
        print(
            f'\x1b[0;30;48;2;{colors_list[idx % len(colors_list)]}m' + 
            tokenizer.decode(t) + 
            '\x1b[0m', 
            end=' '
        )

In [52]:
show_tokens(text, 'bert-base-uncased')

[CLS] english and cap ##iti ##lization [UNK] [UNK] show _ token ##s false none eli ##f = = > = else : two tab ##s : " " three tab ##s : " " 12 . 0 * 50 = 600 . [SEP] 

In [54]:
show_tokens(text, 'bert-base-cased')

[CLS] English and CA ##PI ##TI ##L ##I ##Z ##AT ##ION [UNK] [UNK] show _ token ##s F ##als ##e None el ##if = = > = else : two ta ##bs : " " Three ta ##bs : " " 12 . 0 * 50 = 600 . [SEP] 

In [67]:
show_tokens(text, 'gpt2')


 English  and  CAP IT IL IZ ATION 
 
 � � � � � � 
 show _ t ok ens  False  None  el if  ==  >=  else :  two  tabs :"  "  Three  tabs :  "      " 
 
 12 . 0 * 50 = 600 .   
 

In [73]:
show_tokens(text, "google/flan-t5-xxl")

English and  CAP IT IL IZ ATION  <unk>  <unk> show _ to ken s Fal s e None  e l if = = > = else : two tab s : " " Three tab s : " " 12. 0 * 50 = 600 .  </s> 

In [76]:
show_tokens(text, 'Xenova/gpt-4')

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.



 English  and  CAP IT IL IZATION 

 � � �  � � � 
 show _tokens  False  None  elif  ==  >=  else :  two  tabs :"  "  Three  tabs :  "     "

 12 . 0 * 50 = 600 .  
 

In [88]:
show_tokens(text, "bigcode/starcoder2-15b")
show_tokens(code_text, "bigcode/starcoder2-15b")


 English  and  CAP IT IL IZATION 
 
 � � �   � � 
 show _ tokens  False  None  elif  ==  >=  else :  two  tabs :"    "  Three  tabs :  "     " 

 1 2 . 0 * 5 0 = 6 0 0 .  
 
 def  add _ numbers ( a ,  b ): 
 
 .... #  Add  the  two  numbers  ` a `  and  ` b `. 
 
 .... return  a  +  b 
 

In [83]:
show_tokens(text, "facebook/galactica-1.3b")


 English  and  CAP IT IL IZATION 
 
 � � � �  � � � 
 show _ tokens  False  None  elif   ==   > =  else :  two  t abs : "    "  Three  t abs :   "     " 
 
 1 2 . 0 * 5 0 = 6 0 0 .   
 

In [86]:
show_tokens(text, 'microsoft/Phi-3-mini-4k-instruct')

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


 
 English and C AP IT I LI Z ATION 
 
 � � � �  � � � 
 show _ to kens False None elif == >= else : two tabs :"  " Three tabs : "   " 
 
 1 2 . 0 * 5 0 = 6 0 0 .  
 

### Tokenizer Properties
There are three major groups of design choices that determine how the tokenizer will break down text: the tokenization method, the initialization parameters, and the domain of the data the tokenizer targets.

# Creating Contextualized Word Embeddings with Language Models

In [102]:
from transformers import AutoModel

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained("microsoft/deberta-base")

# Load model
model = AutoModel.from_pretrained("microsoft/deberta-v3-xsmall")

# Tokenize the sentence
tokens = tokenizer('Hello world', return_tensors='pt')

# process tokens
output = model(**tokens)[0]

In [105]:
print(output.shape) # [batch, token #s, embedded vector size]


torch.Size([1, 4, 384])


This particular tokenizer and model operate by adding the [CLS] and [SEP] tokens to the beginning and end of a string. We have 1 batch size, 4 tokens, and the embedding vector size for each token.

In [106]:
# In our case its 1 batch size, the tokenizer made 4 tokens
for token in tokens['input_ids'][0]:
    print(tokenizer.decode(token))
output

[CLS]
Hello
 world
[SEP]


tensor([[[-3.4816,  0.0861, -0.1819,  ..., -0.0612, -0.3911,  0.3017],
         [ 0.1898,  0.3208, -0.2315,  ...,  0.3714,  0.2478,  0.8048],
         [ 0.2071,  0.5036, -0.0485,  ...,  1.2175, -0.2292,  0.8582],
         [-3.4278,  0.0645, -0.1427,  ...,  0.0658, -0.4367,  0.3834]]],
       grad_fn=<NativeLayerNormBackward0>)

## Text Embeddings

In [1]:
from sentence_transformers import SentenceTransformer
# Load model
model = SentenceTransformer("sentence-transformers/all-mpnet-base-v2")

# Convert text to text embedding
vector = model.encode("Best movie ever!")

print(vector.shape)

/Users/brianmorales/miniconda3/envs/thellmbook/lib/python3.10/site-packages/sentence_transformers/cross_encoder/CrossEncoder.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange
W0729 17:47:41.296000 4100 site-packages/torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.
/Users/brianmorales/miniconda3/envs/thellmbook/lib/python3.10/site-packages/huggingface_hub/file_download.py:1150: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


(768,)


In [4]:
import gensim.downloader as api

# Download embeddings (66MB, glove, trained on wikipedia, vector)
# Other options include "word2vec-google-news-300"
model = api.load("glove-wiki-gigaword-50")

In [5]:
model.most_similar([model['king']], topn=11)

[('king', 1.0000001192092896),
 ('prince', 0.8236179351806641),
 ('queen', 0.7839044332504272),
 ('ii', 0.7746230363845825),
 ('emperor', 0.7736247777938843),
 ('son', 0.766719400882721),
 ('uncle', 0.7627150416374207),
 ('kingdom', 0.7542160749435425),
 ('throne', 0.7539914846420288),
 ('brother', 0.7492412328720093),
 ('ruler', 0.7434254288673401)]

# Embeddings for Recommendation Systems

Train a word2vec model to recommend songs

In [1]:
import pandas as pd
from urllib import request

# Get the playlist dataset file
data = request.urlopen('https://storage.googleapis.com/maps-premium/dataset/yes_complete/train.txt')

# Parse the playlist dataset file. Skip the first two lines as 
# they only contain metadata
lines = data.read().decode("utf-8").split('\n')[2:]

# Remove playlist with only one song
playlists = [s.rstrip().split() for s in lines if len(s.split()) > 1]

# Load song metadata
songs_file = request.urlopen('https://storage.googleapis.com/maps-premium/dataset/yes_complete/song_hash.txt')
songs_file = songs_file.read().decode("utf-8").split('\n')
songs = [s.rstrip().split('\t') for s in songs_file]
songs_df = pd.DataFrame(data=songs, columns=['id', 'title', 'artist'])
songs_df = songs_df.set_index('id')

In [2]:
print( 'Playlist #1:\n ', playlists[0], '\n')
print( 'Playlist #2:\n ', playlists[1])

Playlist #1:
  ['0', '1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '20', '21', '22', '23', '24', '25', '26', '27', '28', '29', '30', '31', '32', '33', '34', '35', '36', '37', '38', '39', '40', '41', '2', '42', '43', '44', '45', '46', '47', '48', '20', '49', '8', '50', '51', '52', '53', '54', '55', '56', '57', '25', '58', '59', '60', '61', '62', '3', '63', '64', '65', '66', '46', '47', '67', '2', '48', '68', '69', '70', '57', '50', '71', '72', '53', '73', '25', '74', '59', '20', '46', '75', '76', '77', '59', '20', '43'] 

Playlist #2:
  ['78', '79', '80', '3', '62', '81', '14', '82', '48', '83', '84', '17', '85', '86', '87', '88', '74', '89', '90', '91', '4', '73', '62', '92', '17', '53', '59', '93', '94', '51', '50', '27', '95', '48', '96', '97', '98', '99', '100', '57', '101', '102', '25', '103', '3', '104', '105', '106', '107', '47', '108', '109', '110', '111', '112', '113', '25', '63', '62', '114', '115', '84', '116', '117',

In [3]:
from gensim.models import Word2Vec

# Train our word2vec model
model = Word2Vec(
    playlists, vector_size=32, window=20, negative=50, min_count=1, workers=4, 
)

Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'
Exception ignored in: 'gensim.models.word2vec_inner.our_dot_float'


In [4]:
song_id = 2172

# Ask the model for songs similar to song #2172
model.wv.most_similar(positive=str(song_id))

[('3167', 0.998637318611145),
 ('2976', 0.9984439015388489),
 ('2849', 0.997926652431488),
 ('5586', 0.9969188570976257),
 ('3105', 0.9961821436882019),
 ('5634', 0.9958509802818298),
 ('9995', 0.9954879879951477),
 ('2014', 0.9951347708702087),
 ('2704', 0.9948742985725403),
 ('3094', 0.9946756958961487)]

In [5]:
print(songs_df.iloc[2172])

title     Fade To Black
artist        Metallica
Name: 2172 , dtype: object


In [6]:
import numpy as np

def print_recommendations(song_id): 
    similar_songs = np.array(
        model.wv.most_similar(positive=str(song_id), topn=5)
    )[:, 0]
    return songs_df.iloc[similar_songs]

# Extract recommendations
print_recommendations(2172)

,title,artist
id,,
3167,Unchained,Van Halen
2976,I Don't Know,Ozzy Osbourne
2849,Run To The Hills,Iron Maiden
5586,The Last In Line,Dio
3105,Working Man,Rush
